# ruBERT BIO-tagging Experiments — BERT-0 vs BERT-1

Pipeline: detect spans of textual math-formula descriptions in Russian lecture
transcripts using BIO labels (`O`, `B-MATH`, `I-MATH`).

This notebook runs two experiments with `ai-forever/ruBert-base`:

- **BERT-0** — clean-only baseline, trained on the 185-sentence synthetic `train_clean` split.
- **BERT-1** — curriculum: Stage 1 on the noisy 1261-sentence Mathematicon `train_noisy` split, Stage 2 continues on `train_clean` with a **fresh** optimizer.

Each experiment runs 3 seeds (42, 123, 456). The final cell prints a single
`<executor_output id="E4">` block with all metrics, exactly as specified.

**Before running:** upload/mount the exported HF `DatasetDict` (Arrow format,
folder containing `train_clean`, `train_noisy`, `val`, `test` splits) and set
`DATA_PATH` in the config cell below.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!unzip "/content/drive/MyDrive/linguistic project/hf_dataset.zip" -d "/content/hf_dataset"

Archive:  /content/drive/MyDrive/linguistic project/hf_dataset.zip
replace /content/hf_dataset/hf_dataset/dataset_dict.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [3]:
# Colab ships a `peft` build that's ahead of older `accelerate`/`transformers`
# pins (it imports `accelerate.utils.memory.clear_device_cache`, added later),
# which breaks `from transformers import Trainer` with an ImportError about
# peft/accelerate. We don't use peft/LoRA anywhere in this notebook, so the
# simplest fix is to upgrade transformers/accelerate together and drop peft.
!pip uninstall -y -q peft
!pip install -q -U "transformers>=4.42" "accelerate>=0.31" "datasets>=2.19" "seqeval==1.2.2" scikit-learn

In [4]:
import os, gc, time, inspect, shutil, warnings

import numpy as np
import torch
from datasets import DatasetDict, load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    set_seed,
)
from seqeval.metrics import f1_score as seqeval_f1
from seqeval.scheme import IOB2
from sklearn.metrics import f1_score as sklearn_f1

warnings.filterwarnings("ignore")

# ---- Config ----
MODEL_NAME = "ai-forever/ruBert-base"
FALLBACK_MODEL_NAME = "DeepPavlov/rubert-base-cased"  # escape hatch only, flagged if used
DATA_PATH = "./hf_dataset/hf_dataset"          # <-- Pasha: set to wherever the DatasetDict was uploaded/mounted
MAX_LENGTH = 128
SEEDS = [42, 123, 456]

LABEL_LIST = ["O", "B-MATH", "I-MATH"]
ID2LABEL = {0: "O", 1: "B-MATH", 2: "I-MATH"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
NUM_LABELS = len(LABEL_LIST)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: no GPU detected. Training will be extremely slow / may exceed Colab CPU limits.")

# Flags collected during the run, surfaced verbatim in the final Notes section.
RUN_FLAGS = []

def flag(msg):
    print(f"[FLAG] {msg}")
    RUN_FLAGS.append(msg)

Device: cuda


## 1. Load dataset

In [5]:
assert os.path.exists(DATA_PATH), (
    f"DATA_PATH '{DATA_PATH}' not found. Upload/mount the exported hf_dataset "
    f"folder and update DATA_PATH above before re-running this cell."
)

raw_datasets = load_from_disk(DATA_PATH)
assert isinstance(raw_datasets, DatasetDict), "Expected a DatasetDict at DATA_PATH"

for split in ["train_clean", "train_noisy", "val", "test"]:
    assert split in raw_datasets, f"Missing required split '{split}' in dataset"

for split, ds in raw_datasets.items():
    n_tok = sum(len(ex["tokens"]) for ex in ds)
    print(f"{split:12s} n_sentences={len(ds):5d}  n_tokens={n_tok:6d}  avg_len={n_tok/len(ds):.1f}")

print("\nSample example (train_clean[0]):")
print(raw_datasets["train_clean"][0])

train_clean  n_sentences=  185  n_tokens=  6259  avg_len=33.8
train_noisy  n_sentences= 1261  n_tokens= 21477  avg_len=17.0
val          n_sentences=   30  n_tokens=  1105  avg_len=36.8
test         n_sentences=   50  n_tokens=  2031  avg_len=40.6

Sample example (train_clean[0]):
{'sentence_id': 'synth_0000', 'tokens': ['По', 'свойству', 'логарифмов,', 'логарифм', 'произведения', 'экспоненты', 'в', 'степени', 'квадрата', 'логарифма', 'икс', 'и', 'игрек', 'распадается', 'в', 'сумму,', 'а', 'логарифм', 'от', 'произведения', 'а', 'и', 'бэ', 'равен', 'сумме', 'логарифмов', 'сомножителей.'], 'ner_tags': [0, 0, 0, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 1, 2, 2, 2, 2, 2, 0, 0, 0, 0]}


## 2. Tokenizer & label alignment

In [6]:
def load_tokenizer_and_model_name():
    global MODEL_NAME
    try:
        tok = AutoTokenizer.from_pretrained(MODEL_NAME)
        return tok, MODEL_NAME
    except Exception as e1:
        flag(f"First attempt to load tokenizer for {MODEL_NAME} failed: {e1}. Retrying with trust_remote_code=True.")
        try:
            tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
            return tok, MODEL_NAME
        except Exception as e2:
            flag(
                f"Second attempt for {MODEL_NAME} failed: {e2}. "
                f"Falling back to {FALLBACK_MODEL_NAME} -- RESULTS BELOW USE THE FALLBACK MODEL, NOT ruBert-base."
            )
            tok = AutoTokenizer.from_pretrained(FALLBACK_MODEL_NAME)
            return tok, FALLBACK_MODEL_NAME


tokenizer, ACTIVE_MODEL_NAME = load_tokenizer_and_model_name()
print(f"Active model checkpoint: {ACTIVE_MODEL_NAME}")


def tokenize_and_align_labels(example):
    """Per-example (NOT batched) tokenization + label alignment.
    First subword of each word keeps the original label; continuation
    subwords get -100 so they're ignored by the loss and by metrics."""
    tokenized = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    word_ids = tokenized.word_ids()
    labels = []
    prev_word_id = None
    for wid in word_ids:
        if wid is None:
            labels.append(-100)
        elif wid != prev_word_id:
            labels.append(example["ner_tags"][wid])
        else:
            labels.append(-100)
        prev_word_id = wid
    tokenized["labels"] = labels
    return tokenized


tokenized_datasets = DatasetDict({
    split: ds.map(tokenize_and_align_labels, batched=False, remove_columns=ds.column_names)
    for split, ds in raw_datasets.items()
})

# --- Alignment sanity check on a few examples ---
print("Alignment check (first 3 examples of train_clean):")
for i in range(min(3, len(raw_datasets["train_clean"]))):
    orig = raw_datasets["train_clean"][i]
    enc = tokenizer(orig["tokens"], is_split_into_words=True, truncation=True, max_length=MAX_LENGTH)
    word_ids = enc.word_ids()
    subwords = tokenizer.convert_ids_to_tokens(enc["input_ids"])
    aligned = tokenized_datasets["train_clean"][i]["labels"]
    pairs = [(sw, ID2LABEL.get(lab, "IGN")) for sw, lab in zip(subwords, aligned)]
    print(f"  ex {i}: {pairs}")
    non_ignored = [l for l in aligned if l != -100]
    assert len(non_ignored) == len(orig["ner_tags"]), "Alignment produced wrong number of labeled tokens"
print("Alignment check passed: exactly one labeled subtoken per original word.")

Active model checkpoint: ai-forever/ruBert-base
Alignment check (first 3 examples of train_clean):
  ex 0: [('[CLS]', 'IGN'), ('по', 'O'), ('свои', 'O'), ('##ству', 'IGN'), ('логариф', 'O'), ('##мов', 'IGN'), (',', 'IGN'), ('логариф', 'B-MATH'), ('##м', 'IGN'), ('произведения', 'I-MATH'), ('экспонен', 'I-MATH'), ('##ты', 'IGN'), ('в', 'I-MATH'), ('степени', 'I-MATH'), ('квадрата', 'I-MATH'), ('логариф', 'I-MATH'), ('##ма', 'IGN'), ('икс', 'I-MATH'), ('и', 'I-MATH'), ('игре', 'I-MATH'), ('##к', 'IGN'), ('распадается', 'O'), ('в', 'O'), ('сумму', 'O'), (',', 'IGN'), ('а', 'O'), ('логариф', 'B-MATH'), ('##м', 'IGN'), ('от', 'I-MATH'), ('произведения', 'I-MATH'), ('а', 'I-MATH'), ('и', 'I-MATH'), ('бэ', 'I-MATH'), ('равен', 'O'), ('сумме', 'O'), ('логариф', 'O'), ('##мов', 'IGN'), ('сом', 'O'), ('##но', 'IGN'), ('##жите', 'IGN'), ('##леи', 'IGN'), ('.', 'IGN'), ('[SEP]', 'IGN')]
  ex 1: [('[CLS]', 'IGN'), ('пусть', 'O'), ('даны', 'O'), ('множества', 'O'), ('a', 'O'), (',', 'IGN'), ('b', 'O

## 3. Data collator & metrics

In [7]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)


def _to_tag_sequences(predictions, label_ids):
    """Convert (predictions, label_ids) arrays into lists of string tag
    sequences per sentence, skipping -100 positions."""
    pred_ids = np.argmax(predictions, axis=2)
    true_seqs, pred_seqs = [], []
    for p_row, l_row in zip(pred_ids, label_ids):
        true_seq, pred_seq = [], []
        for p, l in zip(p_row, l_row):
            if l == -100:
                continue
            true_seq.append(ID2LABEL[int(l)])
            pred_seq.append(ID2LABEL[int(p)])
        true_seqs.append(true_seq)
        pred_seqs.append(pred_seq)
    return true_seqs, pred_seqs


def compute_metrics(eval_pred):
    predictions, label_ids = eval_pred
    true_seqs, pred_seqs = _to_tag_sequences(predictions, label_ids)

    # Primary metric: span-level F1, strict IOB2 scheme (seqeval).
    span_f1 = seqeval_f1(true_seqs, pred_seqs, mode="strict", scheme=IOB2)

    # Token-level F1: binary, B-MATH+I-MATH vs O.
    flat_true = [t for seq in true_seqs for t in seq]
    flat_pred = [t for seq in pred_seqs for t in seq]
    bin_true = [0 if t == "O" else 1 for t in flat_true]
    bin_pred = [0 if t == "O" else 1 for t in flat_pred]
    token_f1 = sklearn_f1(bin_true, bin_pred, average="binary", zero_division=0)

    # Exact match: fraction of sentences with a perfectly correct BIO sequence.
    exact_matches = sum(1 for t, p in zip(true_seqs, pred_seqs) if t == p)
    exact_match = exact_matches / len(true_seqs) if true_seqs else 0.0

    return {"span_f1": span_f1, "token_f1": token_f1, "exact_match": exact_match}

## 4. Training utilities

In [8]:
# Support both old (`evaluation_strategy`) and new (`eval_strategy`) TrainingArguments APIs.
_TA_PARAMS = inspect.signature(TrainingArguments.__init__).parameters
_EVAL_KEY = "evaluation_strategy" if "evaluation_strategy" in _TA_PARAMS else "eval_strategy"

# Support both old (`tokenizer`) and new (`processing_class`) Trainer APIs.
_TRAINER_PARAMS = inspect.signature(Trainer.__init__).parameters
_TOKENIZER_KEY = "tokenizer" if "tokenizer" in _TRAINER_PARAMS else "processing_class"


def build_model(checkpoint_path=None):
    """Build a fresh model. If checkpoint_path is None, loads ACTIVE_MODEL_NAME
    from the Hub (with one fallback to FALLBACK_MODEL_NAME if that fails).
    If checkpoint_path is a local path (e.g. a Stage-1 checkpoint), loads from
    there directly -- no Hub fallback applies."""
    global ACTIVE_MODEL_NAME
    path = checkpoint_path or ACTIVE_MODEL_NAME
    try:
        model = AutoModelForTokenClassification.from_pretrained(
            path, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID,
        )
    except Exception as e:
        if checkpoint_path is None and ACTIVE_MODEL_NAME != FALLBACK_MODEL_NAME:
            flag(f"Model load for {path} failed: {e}. Falling back to {FALLBACK_MODEL_NAME}.")
            ACTIVE_MODEL_NAME = FALLBACK_MODEL_NAME
            model = AutoModelForTokenClassification.from_pretrained(
                ACTIVE_MODEL_NAME, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID,
            )
        else:
            raise
    return model


def all_zero_span_f1(log_history):
    vals = [h["eval_span_f1"] for h in log_history if "eval_span_f1" in h]
    return len(vals) > 0 and all(v == 0.0 for v in vals)


def train_one_stage(
    model,
    train_dataset,
    eval_dataset,
    output_dir,
    lr,
    epochs,
    batch_size=16,
    early_stopping_patience=None,
    label_smoothing_factor=0.0,
    seed=42,
    run_label="run",
):
    """Train one stage. Returns (trainer, elapsed_minutes).
    Handles CUDA OOM by retrying once with batch_size=8 + grad_accum=2
    (keeps effective batch size at 16), per the escape-hatch protocol."""
    do_eval = eval_dataset is not None and early_stopping_patience is not None

    def make_args(bsz, grad_accum):
        kwargs = dict(
            output_dir=output_dir,
            learning_rate=lr,
            num_train_epochs=epochs,
            per_device_train_batch_size=bsz,
            per_device_eval_batch_size=32,
            gradient_accumulation_steps=grad_accum,
            weight_decay=0.01,
            label_smoothing_factor=label_smoothing_factor,
            save_strategy="epoch" if do_eval else "no",
            save_total_limit=1,
            load_best_model_at_end=do_eval,
            fp16=(DEVICE == "cuda"),
            seed=seed,
            logging_steps=10,
            report_to="none",
            disable_tqdm=False,
        )
        kwargs[_EVAL_KEY] = "epoch" if do_eval else "no"
        if do_eval:
            kwargs["metric_for_best_model"] = "span_f1"
            kwargs["greater_is_better"] = True
        return TrainingArguments(**kwargs)

    callbacks = [EarlyStoppingCallback(early_stopping_patience=early_stopping_patience)] if early_stopping_patience else []

    bsz, grad_accum = batch_size, 1
    start = time.time()
    attempt = 0
    while True:
        attempt += 1
        args = make_args(bsz, grad_accum)
        trainer_kwargs = dict(
            model=model,
            args=args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset if do_eval else None,
            data_collator=data_collator,
            compute_metrics=compute_metrics if do_eval else None,
            callbacks=callbacks,
        )
        trainer_kwargs[_TOKENIZER_KEY] = tokenizer
        trainer = Trainer(**trainer_kwargs)
        try:
            trainer.train()
            break
        except RuntimeError as e:
            if "out of memory" not in str(e).lower():
                raise
            torch.cuda.empty_cache()
            gc.collect()
            if attempt >= 2:
                raise
            flag(
                f"[{run_label}] CUDA OOM with batch_size={bsz}. Retrying with "
                f"batch_size=8, gradient_accumulation_steps=2 (effective batch size unchanged)."
            )
            bsz, grad_accum = 8, 2

    elapsed_min = (time.time() - start) / 60.0

    if elapsed_min > 30:
        flag(f"[{run_label}] took {elapsed_min:.1f} min (> 30 min). Continuing per protocol.")

    if do_eval and all_zero_span_f1(trainer.state.log_history):
        raise RuntimeError(
            f"[{run_label}] val span-F1 is exactly 0.0 for ALL epochs. "
            f"This indicates a label-alignment or metric-computation bug. Stopping per protocol "
            f"instead of continuing -- inspect tokenize_and_align_labels / compute_metrics before re-running."
        )

    return trainer, elapsed_min


def best_epoch_from_history(log_history, metric="eval_span_f1"):
    best = None
    for h in log_history:
        if metric in h:
            if best is None or h[metric] > best[metric]:
                best = h
    if best is None:
        return None, None
    return best.get("epoch"), best.get(metric)


def cleanup_dir(path):
    shutil.rmtree(path, ignore_errors=True)

## 5. BERT-0: clean-only baseline

In [9]:
bert0_results = []  # one dict per seed
bert0_best_overall = {"seed": None, "epoch": None, "val_span_f1": -1.0}

for seed in SEEDS:
    print(f"\n=== BERT-0 | seed={seed} ===")
    set_seed(seed)
    model = build_model()
    out_dir = f"./tmp_bert0_seed{seed}"

    trainer, elapsed_min = train_one_stage(
        model=model,
        train_dataset=tokenized_datasets["train_clean"],
        eval_dataset=tokenized_datasets["val"],
        output_dir=out_dir,
        lr=2e-5,
        epochs=10,
        batch_size=16,
        early_stopping_patience=3,
        label_smoothing_factor=0.0,
        seed=seed,
        run_label=f"BERT-0 seed{seed}",
    )

    val_metrics = trainer.evaluate(tokenized_datasets["val"])
    test_metrics = trainer.evaluate(tokenized_datasets["test"])

    best_epoch, best_val_f1 = best_epoch_from_history(trainer.state.log_history)
    if best_val_f1 is not None and best_val_f1 > bert0_best_overall["val_span_f1"]:
        bert0_best_overall = {"seed": seed, "epoch": best_epoch, "val_span_f1": best_val_f1}

    bert0_results.append({
        "seed": seed,
        "val_span_f1": val_metrics["eval_span_f1"],
        "test_span_f1": test_metrics["eval_span_f1"],
        "test_token_f1": test_metrics["eval_token_f1"],
        "test_exact_match": test_metrics["eval_exact_match"],
        "train_time_min": elapsed_min,
    })
    print(bert0_results[-1])

    cleanup_dir(out_dir)
    del trainer, model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


=== BERT-0 | seed=42 ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: ai-forever/ruBert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ign

Epoch,Training Loss,Validation Loss,Span F1,Token F1,Exact Match
1,0.738880,0.785100,0.000000,0.783523,0.000000
2,0.379582,0.782910,0.017544,0.792315,0.000000
3,0.235543,0.812030,0.061069,0.798568,0.033333
4,0.150839,0.887775,0.150685,0.804270,0.100000
5,0.085782,1.013188,0.142857,0.800681,0.066667
6,0.066025,0.986432,0.165605,0.795435,0.100000
7,0.050284,0.991351,0.143713,0.789615,0.100000
8,0.033395,1.083575,0.156627,0.797927,0.066667
9,0.052403,1.060809,0.156627,0.793988,0.066667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Span F1,Token F1,Exact Match
0.039415,0.910592,8,0.159091,0.806957,0.066667


Training Loss,Validation Loss,Epoch,Span F1,Token F1,Exact Match
0.039415,0.790497,8,0.198630,0.830645,0.100000


{'seed': 456, 'val_span_f1': 0.1590909090909091, 'test_span_f1': 0.19863013698630136, 'test_token_f1': 0.8306451612903226, 'test_exact_match': 0.1, 'train_time_min': 2.5163894732793173}


## 6. BERT-1: curriculum (Mathematicon → synthetic)

In [10]:
bert1_results = []
bert1_best_overall = {"seed": None, "epoch": None, "val_span_f1": -1.0}

for seed in SEEDS:
    print(f"\n=== BERT-1 | seed={seed} | Stage 1 (noisy) ===")
    set_seed(seed)
    model = build_model()
    stage1_dir = f"./tmp_bert1_stage1_seed{seed}"

    trainer_s1, stage1_min = train_one_stage(
        model=model,
        train_dataset=tokenized_datasets["train_noisy"],
        eval_dataset=None,
        output_dir=stage1_dir,
        lr=3e-5,
        epochs=3,
        batch_size=16,
        early_stopping_patience=None,
        label_smoothing_factor=0.1,
        seed=seed,
        run_label=f"BERT-1 seed{seed} stage1",
    )

    # Save Stage-1 weights to disk so Stage 2 loads a genuinely fresh
    # model + fresh optimizer/Trainer (no carried-over optimizer state).
    stage1_ckpt = f"./tmp_bert1_stage1_ckpt_seed{seed}"
    trainer_s1.save_model(stage1_ckpt)
    cleanup_dir(stage1_dir)
    del trainer_s1, model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    print(f"=== BERT-1 | seed={seed} | Stage 2 (clean, fresh optimizer) ===")
    model_s2 = build_model(checkpoint_path=stage1_ckpt)
    stage2_dir = f"./tmp_bert1_stage2_seed{seed}"

    trainer_s2, stage2_min = train_one_stage(
        model=model_s2,
        train_dataset=tokenized_datasets["train_clean"],
        eval_dataset=tokenized_datasets["val"],
        output_dir=stage2_dir,
        lr=1.5e-5,
        epochs=15,
        batch_size=16,
        early_stopping_patience=3,
        label_smoothing_factor=0.0,
        seed=seed,
        run_label=f"BERT-1 seed{seed} stage2",
    )

    val_metrics = trainer_s2.evaluate(tokenized_datasets["val"])
    test_metrics = trainer_s2.evaluate(tokenized_datasets["test"])

    best_epoch, best_val_f1 = best_epoch_from_history(trainer_s2.state.log_history)
    if best_val_f1 is not None and best_val_f1 > bert1_best_overall["val_span_f1"]:
        bert1_best_overall = {"seed": seed, "epoch": best_epoch, "val_span_f1": best_val_f1}

    total_min = stage1_min + stage2_min

    bert1_results.append({
        "seed": seed,
        "val_span_f1": val_metrics["eval_span_f1"],
        "test_span_f1": test_metrics["eval_span_f1"],
        "test_token_f1": test_metrics["eval_token_f1"],
        "test_exact_match": test_metrics["eval_exact_match"],
        "stage1_time_min": stage1_min,
        "stage2_time_min": stage2_min,
        "train_time_min": total_min,
    })
    print(bert1_results[-1])

    cleanup_dir(stage2_dir)
    cleanup_dir(stage1_ckpt)
    del trainer_s2, model_s2
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


=== BERT-1 | seed=42 | Stage 1 (noisy) ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: ai-forever/ruBert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ign

Step,Training Loss
10,0.908858
20,0.756277
30,0.695984
40,0.682405
50,0.629148
60,0.578290
70,0.565270
80,0.536142
90,0.486019
100,0.488482


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Span F1,Token F1,Exact Match
0.083644,0.848948,7,0.213483,0.784452,0.033333


Training Loss,Validation Loss,Epoch,Span F1,Token F1,Exact Match
0.083644,0.758112,7,0.201389,0.818223,0.040000


{'seed': 456, 'val_span_f1': 0.2134831460674157, 'test_span_f1': 0.2013888888888889, 'test_token_f1': 0.8182232346241458, 'test_exact_match': 0.04, 'stage1_time_min': 0.3474058389663696, 'stage2_time_min': 4.9064116398493445, 'train_time_min': 5.253817478815714}


## 7. Aggregate & report

In [12]:
def mean_std(values):
    arr = np.array(values, dtype=float)
    return float(arr.mean()), float(arr.std(ddof=0))


def fmt(v):
    return f"{v:.4f}"


def fmt_pm(mean, std):
    return f"{mean:.4f}±{std:.4f}"


def aggregate(results):
    return {
        "val_span_f1": mean_std([r["val_span_f1"] for r in results]),
        "test_span_f1": mean_std([r["test_span_f1"] for r in results]),
        "test_token_f1": mean_std([r["test_token_f1"] for r in results]),
        "test_exact_match": mean_std([r["test_exact_match"] for r in results]),
    }


agg0 = aggregate(bert0_results)
agg1 = aggregate(bert1_results)

bert0_rows = "\n".join(
    f"| {r['seed']} | {fmt(r['val_span_f1'])} | {fmt(r['test_span_f1'])} | {fmt(r['test_token_f1'])} | {fmt(r['test_exact_match'])} |"
    for r in bert0_results
)
bert1_rows = "\n".join(
    f"| {r['seed']} | {fmt(r['val_span_f1'])} | {fmt(r['test_span_f1'])} | {fmt(r['test_token_f1'])} | {fmt(r['test_exact_match'])} |"
    for r in bert1_results
)

bert0_time_avg = float(np.mean([r["train_time_min"] for r in bert0_results]))
bert1_time_avg = float(np.mean([r["train_time_min"] for r in bert1_results]))
bert1_stage1_avg = float(np.mean([r["stage1_time_min"] for r in bert1_results]))
bert1_stage2_avg = float(np.mean([r["stage2_time_min"] for r in bert1_results]))

delta = agg1["test_span_f1"][0] - agg0["test_span_f1"][0]
delta_str = f"+{delta:.4f}" if delta >= 0 else f"{delta:.4f}"

notes = "; ".join(RUN_FLAGS) if RUN_FLAGS else "No issues encountered during training."
if ACTIVE_MODEL_NAME != MODEL_NAME:
    notes = f"USED FALLBACK MODEL {ACTIVE_MODEL_NAME} instead of {MODEL_NAME}. " + notes

output_block = f"""<executor_output id="E4">
## ruBERT Experiment Results

### BERT-0: Clean-only baseline

| Seed | Val Span-F1 | Test Span-F1 | Test Token-F1 | Test Exact Match |
|------|-------------|--------------|---------------|------------------|
{bert0_rows}
| Mean±Std | {fmt_pm(*agg0['val_span_f1'])} | {fmt_pm(*agg0['test_span_f1'])} | {fmt_pm(*agg0['test_token_f1'])} | {fmt_pm(*agg0['test_exact_match'])} |

Best checkpoint: seed={bert0_best_overall['seed']}, epoch={bert0_best_overall['epoch']}, val_span_f1={fmt(bert0_best_overall['val_span_f1'])}
Training time per seed (approx): {bert0_time_avg:.0f} min

### BERT-1: Curriculum (Mathematicon → synthetic)

| Seed | Val Span-F1 | Test Span-F1 | Test Token-F1 | Test Exact Match |
|------|-------------|--------------|---------------|------------------|
{bert1_rows}
| Mean±Std | {fmt_pm(*agg1['val_span_f1'])} | {fmt_pm(*agg1['test_span_f1'])} | {fmt_pm(*agg1['test_token_f1'])} | {fmt_pm(*agg1['test_exact_match'])} |

Best checkpoint: seed={bert1_best_overall['seed']}, epoch={bert1_best_overall['epoch']}, val_span_f1={fmt(bert1_best_overall['val_span_f1'])}
Training time per seed (approx): {bert1_time_avg:.0f} min (stage1: {bert1_stage1_avg:.0f} min, stage2: {bert1_stage2_avg:.0f} min)

### Comparison

| Model  | Test Span-F1 (mean±std) | Δ vs BERT-0 |
|--------|--------------------------|-------------|
| BERT-0 | {fmt_pm(*agg0['test_span_f1'])} | — |
| BERT-1 | {fmt_pm(*agg1['test_span_f1'])} | {delta_str} |

Notes: {notes}
</executor_output>"""

print(output_block)

<executor_output id="E4">
## ruBERT Experiment Results

### BERT-0: Clean-only baseline

| Seed | Val Span-F1 | Test Span-F1 | Test Token-F1 | Test Exact Match |
|------|-------------|--------------|---------------|------------------|
| 42 | 0.1656 | 0.1882 | 0.8314 | 0.0800 |
| 123 | 0.1745 | 0.1928 | 0.8246 | 0.1200 |
| 456 | 0.1591 | 0.1986 | 0.8306 | 0.1000 |
| Mean±Std | 0.1664±0.0063 | 0.1932±0.0043 | 0.8289±0.0031 | 0.1000±0.0163 |

Best checkpoint: seed=456, epoch=8.0, val_span_f1=0.1986
Training time per seed (approx): 3 min

### BERT-1: Curriculum (Mathematicon → synthetic)

| Seed | Val Span-F1 | Test Span-F1 | Test Token-F1 | Test Exact Match |
|------|-------------|--------------|---------------|------------------|
| 42 | 0.1839 | 0.1919 | 0.8259 | 0.0800 |
| 123 | 0.1714 | 0.1600 | 0.8126 | 0.0400 |
| 456 | 0.2135 | 0.2014 | 0.8182 | 0.0400 |
| Mean±Std | 0.1896±0.0176 | 0.1844±0.0177 | 0.8189±0.0055 | 0.0533±0.0189 |

Best checkpoint: seed=456, epoch=4.0, val_span_f1=0.2